# Aliado Libre — evaluación de los 4 modelos en GPU (Colab)

Genera las respuestas de los 4 modelos GGUF a las 150 preguntas de prueba, usando la GPU de Colab en vez de tu CPU (10-50x más rápido). El juicio de cuáles respuestas son correctas se hace después, local, con las respuestas que este notebook descarga.

**No necesita PyTorch** — `llama-cpp-python` es un motor en C++ aparte, no usa `torch` para nada (a diferencia de los notebooks de entrenamiento). Solo necesita compilarse o instalarse con soporte CUDA, que este notebook maneja solo.

**Antes de correr:** `Entorno de ejecución` → `Cambiar tipo de entorno de ejecución` → GPU (T4 alcanza).

**Todo se lee directo de tu Google Drive** — no hace falta subir nada manualmente, la carpeta ya está en `00_Programas y macros/aliado_libre_eval/` dentro de tu Drive, solo espera a que termine de sincronizar antes de correr.

**Pasos:**
1. `Entorno de ejecución` → `Ejecutar todas`.
2. Cuando pida autorizar Google Drive, acepta con tu misma cuenta.
3. Al final se descarga `respuestas_gpu.json` — muévelo a `finetune/eval/respuestas_gpu.json` en tu proyecto local (en `C:\Users\Lenovo\aliado-libre`, NO en Drive).

In [ ]:
# Instalación robusta con soporte GPU: intenta primero un wheel precompilado
# (rápido, sin riesgo de fallo de compilación), y si no hay uno disponible
# para la versión de CUDA de esta sesión de Colab, compila desde código con
# soporte CUDA explícito. Cualquiera de los dos caminos deja llama-cpp-python
# con aceleración GPU funcionando.
import subprocess
import sys

def _correr(cmd):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True).returncode

codigo = _correr(
    "pip install -q llama-cpp-python "
    "--extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124"
)

if codigo != 0:
    print("\nEl wheel precompilado no aplicó, compilando desde código fuente con CUDA (puede tardar unos minutos)...")
    import os
    os.environ["CMAKE_ARGS"] = "-DGGML_CUDA=on"
    codigo = _correr("pip install -q llama-cpp-python --upgrade --force-reinstall --no-cache-dir")
    if codigo != 0:
        raise RuntimeError(
            "No se pudo instalar llama-cpp-python con soporte CUDA por ninguno de los dos caminos. "
            "Revisa el mensaje de error de arriba."
        )

print("\nInstalación lista.")

In [ ]:
from llama_cpp import Llama, llama_cpp

# verificación dura: si por alguna razón el wheel instalado NO trae soporte
# CUDA (ej. cayó al wheel genérico de CPU), esto avisa ANTES de perder tiempo
# corriendo los 4 modelos a velocidad de CPU sin darte cuenta.
soporta_gpu = llama_cpp.llama_supports_gpu_offload()
print(f"Soporte GPU en llama-cpp-python: {soporta_gpu}")
assert soporta_gpu, (
    "llama-cpp-python se instaló SIN soporte GPU — revisa que el tipo de entorno de "
    "ejecución sea GPU (Entorno de ejecución > Cambiar tipo de entorno de ejecución) "
    "y reinicia el entorno antes de reintentar."
)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CARPETA = '/content/drive/MyDrive/00_Programas y macros/aliado_libre_eval'
import os
assert os.path.isdir(CARPETA), (
    f"No existe {CARPETA} — espera a que Google Drive termine de sincronizar esa "
    "carpeta en tu PC antes de correr esta celda."
)
print('Archivos en la carpeta:')
for f in sorted(os.listdir(CARPETA)):
    print(' -', f)

In [ ]:
import json

with open(f'{CARPETA}/prompts_precalculados.json', encoding='utf-8') as f:
    casos = json.load(f)
casos = [c for c in casos if c['prompt'] is not None]
print(f'{len(casos)} preguntas con prompt (con fragmentos) para evaluar.')
assert len(casos) > 0, "No hay casos con prompt — revisa prompts_precalculados.json"

In [ ]:
import glob

modelos_gguf = sorted(glob.glob(f'{CARPETA}/*.gguf'))
print(f'{len(modelos_gguf)} modelos encontrados:')
for m in modelos_gguf:
    print(' -', m)
assert len(modelos_gguf) > 0, f"No hay archivos .gguf en {CARPETA}"

In [ ]:
resultados_por_modelo = {}

for ruta_modelo in modelos_gguf:
    nombre = ruta_modelo.split('/')[-1]
    print(f'\n=== {nombre} ===')
    # n_gpu_layers=-1: todas las capas a GPU. n_ctx=8192: mismo tamaño que se usó
    # local, evita el crash de contexto que ya vimos con 4096.
    modelo = Llama(model_path=ruta_modelo, n_ctx=8192, n_gpu_layers=-1, verbose=False)

    respuestas = []
    for i, caso in enumerate(casos, 1):
        try:
            salida = modelo(caso['prompt'], max_tokens=400, temperature=0.2, stop=['Pregunta:', '###'])
            respuesta = salida['choices'][0]['text'].strip()
        except ValueError as e:
            respuesta = f"(error: {e})"
        respuestas.append(
            {
                'pregunta': caso['pregunta'],
                'respuesta_esperada': caso['respuesta_esperada'],
                'es_negativo': caso['es_negativo'],
                'fragmentos': caso['fragmentos'],
                'respuesta_modelo': respuesta,
            }
        )
        if i % 20 == 0:
            print(f'  {i}/{len(casos)}')

    resultados_por_modelo[nombre] = respuestas

    # guarda parcial tras cada modelo por si algo falla a mitad de los 4 —
    # no se pierde el trabajo ya hecho de los modelos anteriores
    with open('respuestas_gpu.json', 'w', encoding='utf-8') as f:
        json.dump(resultados_por_modelo, f, ensure_ascii=False, indent=2)

    del modelo

print('\nTodos los modelos evaluados.')

In [ ]:
from google.colab import files
files.download('respuestas_gpu.json')